In [ ]:
import os
import sys
import subprocess

# =====================================================================
# 1. AUTO-INSTALLER (For Google Colab)
# =====================================================================
def install_dependencies():
    print("Setting up the AI environment... This will take about 2-3 minutes.")
    packages = [
        "torch", "torchvision", "torchaudio",
        "git+https://github.com/facebookresearch/sam2.git",
        "iopath", "hydra-core", "tifffile", "gradio", "opencv-python", "matplotlib"
    ]
    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
    print("Setup Complete!")

try:
    import gradio as gr
    from sam2.build_sam import build_sam2_video_predictor
except ImportError:
    install_dependencies()
    import gradio as gr
    from sam2.build_sam import build_sam2_video_predictor

import shutil
import cv2
import numpy as np
import tifffile
import torch
import urllib.request

# =====================================================================
# 2. MODEL DOWNLOAD & INITIALIZATION
# =====================================================================
SAM2_CHECKPOINT = "sam2.1_hiera_small.pt"
CHECKPOINT_URL = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt"
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_s.yaml"
TEMP_DIR = "./temp_gradio_frames"

if not os.path.exists(SAM2_CHECKPOINT):
    print("Downloading SAM 2 AI Brain...")
    urllib.request.urlretrieve(CHECKPOINT_URL, SAM2_CHECKPOINT)

print("Loading SAM 2 into GPU...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True

predictor = build_sam2_video_predictor(SAM2_CONFIG, SAM2_CHECKPOINT, device=device)

# =====================================================================
# 3. GRADIO WEB INTERFACE LOGIC
# =====================================================================
def render_frame_with_points(frames, frame_idx, all_prompts):
    """Helper to draw a specific frame and overlay any points saved for it."""
    if frames is None: return None

    raw_frame = frames[frame_idx]
    if raw_frame.dtype != np.uint8:
        raw_frame = cv2.normalize(raw_frame, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    else:
        raw_frame = raw_frame.copy()

    if len(raw_frame.shape) == 2:
        display_img = cv2.cvtColor(raw_frame, cv2.COLOR_GRAY2RGB)
    else:
        display_img = raw_frame.copy()

    if frame_idx in all_prompts:
        pts = all_prompts[frame_idx]['pts']
        lbls = all_prompts[frame_idx]['lbls']
        for p, l in zip(pts, lbls):
            color = (0, 255, 0) if l == 1 else (255, 0, 0)
            cv2.circle(display_img, tuple(p), 6, color, -1)
            cv2.circle(display_img, tuple(p), 6, (255, 255, 255), 2)

    return display_img

def process_video_upload(file_path):
    if file_path is None:
        return None, None, {}, gr.update(maximum=1, value=0, interactive=False)

    frames = tifffile.imread(file_path)
    if frames.ndim == 2: frames = np.expand_dims(frames, axis=0)

    all_prompts = {}
    rendered_frame = render_frame_with_points(frames, 0, all_prompts)

    return frames, rendered_frame, all_prompts, gr.update(maximum=len(frames)-1, value=0, interactive=True)

def change_frame(frame_idx, frames, all_prompts):
    return render_frame_with_points(frames, frame_idx, all_prompts)

def add_point(evt: gr.SelectData, pt_type, frame_idx, frames, all_prompts):
    if frames is None: return None, all_prompts

    x, y = evt.index
    label = 1 if pt_type == "Foreground (Fish)" else 0

    if frame_idx not in all_prompts:
        all_prompts[frame_idx] = {'pts': [], 'lbls': []}

    all_prompts[frame_idx]['pts'].append([x, y])
    all_prompts[frame_idx]['lbls'].append(label)

    rendered_frame = render_frame_with_points(frames, frame_idx, all_prompts)
    return rendered_frame, all_prompts

def clear_current_frame(frame_idx, frames, all_prompts):
    if frame_idx in all_prompts:
        del all_prompts[frame_idx]
    rendered = render_frame_with_points(frames, frame_idx, all_prompts)
    return rendered, all_prompts

def clear_all_frames(frame_idx, frames, all_prompts):
    all_prompts.clear()
    rendered = render_frame_with_points(frames, frame_idx, all_prompts)
    return rendered, all_prompts

def run_tracking(frames, all_prompts):
    if frames is None or len(all_prompts) == 0:
        return None, None

    if os.path.exists(TEMP_DIR): shutil.rmtree(TEMP_DIR)
    os.makedirs(TEMP_DIR)

    for i, frame in enumerate(frames):
        f_norm = cv2.normalize(frame, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8) if frame.dtype != np.uint8 else frame.copy()
        cv2.imwrite(os.path.join(TEMP_DIR, f"{i:05d}.jpg"), f_norm)

    inference_state = predictor.init_state(video_path=TEMP_DIR)

    for f_idx, prompt_data in all_prompts.items():
        pts = prompt_data.get('pts', [])
        lbls = prompt_data.get('lbls', [])
        if len(pts) > 0:
            predictor.add_new_points_or_box(
                inference_state=inference_state, frame_idx=f_idx, obj_id=1,
                points=np.array(pts, dtype=np.float32), labels=np.array(lbls, dtype=np.int32)
            )

    video_segments = {}

    # Track Forward
    for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
        video_segments[out_frame_idx] = (out_mask_logits[0] > 0.0).cpu().numpy().squeeze()

    # Track Backward
    for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state, reverse=True):
        video_segments[out_frame_idx] = (out_mask_logits[0] > 0.0).cpu().numpy().squeeze()

    blackout_stack = []
    for i, frame in enumerate(frames):
        f_norm = cv2.normalize(frame, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8) if frame.dtype != np.uint8 else frame.copy()
        f_inv = cv2.bitwise_not(f_norm)
        mask = video_segments.get(i, np.zeros(frame.shape[:2], dtype=bool))

        blackout = np.zeros_like(f_inv)
        blackout[mask] = f_inv[mask]
        blackout_stack.append(blackout)

    # --- Generate Outputs ---
    tif_output_path = "Sanitized_Blackout_Video.tif"
    avi_output_path = "Sanitized_Blackout_Video.avi"

    # 1. Save TIF
    tifffile.imwrite(tif_output_path, np.array(blackout_stack), photometric='minisblack')

    # 2. Save AVI (Converting frames to 3-channel BGR so all media players can read it)
    height, width = blackout_stack[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    out_avi = cv2.VideoWriter(avi_output_path, fourcc, 15.0, (width, height)) # 15 FPS

    for frame in blackout_stack:
        if len(frame.shape) == 2:
            frame_bgr = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
        else:
            frame_bgr = frame
        out_avi.write(frame_bgr)
    out_avi.release()

    shutil.rmtree(TEMP_DIR)

    return tif_output_path, avi_output_path

# =====================================================================
# 4. BUILD THE GRADIO WEB UI
# =====================================================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🐟 SAM 2 Zebrafish Tracker (Multi-Frame Edition)")
    gr.Markdown("Upload a video. Slide through the video and place tracking points on **multiple different frames**. Click Process to generate both TIF data and an AVI video preview!")

    with gr.Row():
        with gr.Column(scale=1):
            video_upload = gr.File(label="1. Upload .TIF Video Stack", file_types=[".tif", ".tiff"])
            frame_slider = gr.Slider(minimum=0, maximum=1, step=1, value=0, label="2. Navigate Frames", interactive=False)
            point_type = gr.Radio(["Foreground (Fish)", "Background (Triangle)"], value="Foreground (Fish)", label="3. Select Click Type")

            with gr.Row():
                btn_clear_frame = gr.Button("Clear This Frame", size="sm")
                btn_clear_all = gr.Button("Clear All Frames", size="sm")

            btn_run = gr.Button("4. Process Video", variant="primary")

            # Now featuring TWO download blocks!
            with gr.Row():
                tif_download = gr.File(label="Download Data (.TIF)", interactive=False)
                avi_download = gr.File(label="Download Video (.AVI)", interactive=False)

        with gr.Column(scale=2):
            image_display = gr.Image(label="Click directly on the image to add tracking points!", interactive=False)

    state_frames = gr.State(None)
    state_all_prompts = gr.State({})

    video_upload.upload(
        fn=process_video_upload, inputs=[video_upload],
        outputs=[state_frames, image_display, state_all_prompts, frame_slider]
    )
    frame_slider.change(
        fn=change_frame, inputs=[frame_slider, state_frames, state_all_prompts],
        outputs=[image_display]
    )
    image_display.select(
        fn=add_point, inputs=[point_type, frame_slider, state_frames, state_all_prompts],
        outputs=[image_display, state_all_prompts]
    )
    btn_clear_frame.click(
        fn=clear_current_frame, inputs=[frame_slider, state_frames, state_all_prompts],
        outputs=[image_display, state_all_prompts]
    )
    btn_clear_all.click(
        fn=clear_all_frames, inputs=[frame_slider, state_frames, state_all_prompts],
        outputs=[image_display, state_all_prompts]
    )
    btn_run.click(
        fn=run_tracking, inputs=[state_frames, state_all_prompts],
        # Update output target to hit both download boxes
        outputs=[tif_download, avi_download]
    )

if __name__ == "__main__":
    demo.launch(share=True, debug=True)

Loading SAM 2 into GPU...


/tmp/ipykernel_1298/3555516566.py:193: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3c2e8227d532ddb333.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/di

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3c2e8227d532ddb333.gradio.live
